# 用 Phi-3 學「提示工程」：基本生成、聊天模板、temperature、top_p、情境學習、連鎖提示
---

## 這堂課你會學到什麼

1. 用 `pipeline("text-generation")` 載入並使用一個 LLM
2. 看懂**訊息格式**（`role` / `content`）與**聊天模板**（`apply_chat_template`）
3. 用 **temperature**、**top_p** 控制「模型有多敢亂講」
4. **情境學習（in-context learning）**：用範例教模型
5. **連鎖提示（prompt chaining）**：把上一個輸出變成下一個輸入

## 0. 環境準備

```bash
pip install torch transformers accelerate
```

- 程式使用 `device_map = "cuda"`，需要 **NVIDIA 顯示卡 + CUDA**。
- Phi-3-mini 以 16 位元載入，權重約 7.6 GB（我用參數量估算的）；8 GB 的顯卡可能放不下，12 GB 以上比較穩。
- 第一次執行會從 Hugging Face 下載模型，需要網路與足夠硬碟空間。
- 如果你的 transformers 是新版，載入時可能出現 `torch_dtype is deprecated! Use dtype instead!` 的警告：舊參數名稱仍然可以用，只是建議改名。

## 1. 先看整體：LLM 是怎麼「說話」的？

你給模型一段文字，它做的事只有一件：**預測下一個 token**，然後一個一個接下去。

```
 你的訊息（list of dict）
    │
    ▼ ① 套用聊天模板      tokenizer.apply_chat_template
 一段帶有角色標記的文字
    │
    ▼ ② 分詞              tokenizer
 token id（一串數字）
    │
    ▼ ③ 生成              model.generate  ← temperature、top_p 在這裡起作用
 新的 token id
    │
    ▼ ④ 解碼              tokenizer.decode
 文字答案
```

`pipeline("text-generation")` 就是把 ①②③④ **包成一個動作**：你只要丟訊息進去，它就替你走完全部步驟。這份教材前半段用 pipeline，同時也會讓你「看見」其中的第 ① 步。

> 📌 **本節重點**
> - LLM 一次預測一個 token，反覆進行直到結束。
> - pipeline = 套模板 + 分詞 + 生成 + 解碼，一次完成。
> - temperature、top_p 影響的是「第 ③ 步怎麼從候選 token 裡挑一個」。

## 2. 匯入套件

| 套件 | 用途 |
|---|---|
| `torch` | PyTorch，模型運算的底層（這份程式沒有直接用到，但 transformers 需要它） |
| `AutoModelForCausalLM` | 自動載入「因果語言模型」（一個字一個字往後預測的模型） |
| `AutoTokenizer` | 自動載入對應的分詞器 |
| `pipeline` | 把模型 + 分詞器包成好用的生成工具 |

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

## 3. 載入模型

```python
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map = "cuda",
    torch_dtype = "auto",
)
```

| 參數 | 意思 |
|---|---|
| `"microsoft/Phi-3-mini-4k-instruct"` | Hugging Face 上的模型名稱；`instruct` 表示做過**指令微調**，會「回答問題」；`4k` 是上下文長度 |
| `device_map = "cuda"` | 直接把模型放到 GPU 上 |
| `torch_dtype = "auto"` | 依模型設定自動挑數值精度（通常是 16 位元），比預設的 32 位元省一半顯存 |

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map = "cuda",
    torch_dtype = "auto",
)

## 4. 載入分詞器

模型只懂數字，分詞器（tokenizer）負責「文字 ⇄ token id」的互相轉換。**模型和分詞器一定要配對**，所以兩邊用同一個名稱。

這份分詞器還有另一個重要的身份：它**記得這個模型的聊天模板**（第 7 節會用到）。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)

## 5. 建立生成管線 `pipe`

| 參數 | 意思 |
|---|---|
| `"text-generation"` | 任務類型：文字生成 |
| `model`、`tokenizer` | 剛剛載入的兩個零件 |
| `return_full_text = False` | 只回傳**新生成的部分**，不把你的提示詞也一起回傳 |
| `max_new_tokens = 500` | 最多生成 500 個新 token，避免無限講下去 |
| `do_sample = False` | **不取樣**：每一步都選機率最高的 token（貪婪解碼）。結果穩定、可重現 |

In [ ]:
pipe = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False,
    max_new_tokens = 500,
    do_sample = False,
)

> 📌 **本節重點**
> - `do_sample = False` 是這份教材的「基準線」：同樣的輸入 → 同樣的輸出。
> - 後面 temperature、top_p 的示範，都要先**打開取樣**（`do_sample=True`）才有意義。

## 6. 訊息格式與基本生成

### 訊息（message）長什麼樣？

```python
message = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
```

- 是一個 **list**，裡面每個元素是一個 **dict**，代表一則訊息。
- `role` 是說話的人：`"user"`（你）、`"assistant"`（模型）、`"system"`（給模型的整體規則，這份程式沒用到）。
- `content` 是說話的內容。

### 生成

`pipe(message)` 回傳的是 **list of dict**，所以取答案要寫 `output[0]["generated_text"]`：`[0]` 是第一個（也是唯一一個）結果，`["generated_text"]` 是裡面的文字。

因為 `do_sample = False`，每次執行結果都會一樣。

In [ ]:
message = [
    {
        "role" : "user",
        "content" : "Create a funny joke about chickens."
    }
]

In [ ]:
output = pipe(message)
print(output[0]["generated_text"])

> 📌 **本節重點**
> - 訊息 = list of dict，每則有 `role` 和 `content`。
> - 傳**訊息 list** 給 pipeline，它會自動套用聊天模板（下一節就來看模板長什麼樣）。

## 7. 聊天模板：模型實際「看到」的文字

模型並不是真的看到你的 list。它看到的是一段**帶有特殊角色標記的文字**：

```
<|user|>
Create a funny joke about chickens.<|end|>
```

不同模型的標記不一樣，所以要由**分詞器**依該模型的模板產生，這就是 `apply_chat_template` 的工作。

| 參數 | 意思 |
|---|---|
| `message` | 要轉換的訊息 list |
| `tokenize=False` | 回傳**字串**，方便 `print` 來看；不寫的話預設是轉成 token id（數字），就看不到文字了 |

### 兩個容易搞混的地方

1. **`pipe.tokenizer` 和 `tokenizer` 是同一個物件**（pipeline 會保存你傳進去的分詞器），所以用哪個都可以。
2. **這裡只是「看」，不是生成的必要步驟。** `pipe(message)` 內部已經自動套過模板了。你印出來的這份，只是讓你知道模型收到什麼。
   - 細節：pipeline 內部呼叫時會加上 `add_generation_prompt=True`（在結尾補上「輪到助理回答」的標記），而你這行沒有加，所以印出來的會比模型實際收到的**少結尾的助理標記**。想看到完全一樣的版本，可以自己多加這個參數試試看。

In [ ]:
#===================================================================
#訊息轉換為特定的提示詞模板
prompt = pipe.tokenizer.apply_chat_template(message, tokenize=False) 
print(prompt)
#===================================================================

> 📌 **本節重點**
> - 訊息 list →（聊天模板）→ 帶角色標記的字串 →（分詞）→ token id。
> - 傳訊息 list 給 pipeline：它代勞。自己用 `model.generate()`：要自己套模板。
> - 想看文字 → `tokenize=False`。

## 8. Temperature：控制「敢不敢選冷門的字」

模型每一步會給所有候選 token 一個機率。`do_sample=False` 時永遠選最高的；`do_sample=True` 時則**依機率抽籤**。

**temperature** 在抽籤之前，先把機率分佈「拉平」或「拉尖」：

| temperature | 效果 |
|---|---|
| 小於 1（如 0.2） | 分佈更尖，**更保守**，高機率的字更容易被選中 |
| 等於 1 | **不調整**，維持模型原本的機率 |
| 大於 1（如 1.5） | 分佈更平，**更敢冒險**，冷門的字比較有機會被選到 |

原程式的註解寫「提高選擇不太可能的字元的機率」，這是 temperature **大於 1** 時的效果；程式裡設的是 `temperature=1`，等於維持原本分佈。

> ⚠️ **注意：** 如果 `do_sample=False`，`temperature` 會被忽略（transformers 會警告你參數沒作用）。所以程式裡一定要同時寫 `do_sample=True`。

In [ ]:
#===================================================================
#Temperatur提高選擇不太可能的字元的機率
output = pipe(message, do_sample=True, temperature=1)
print(output[0]["generated_text"])
#===================================================================

> 📌 **本節重點**
> - `do_sample=True` 才會抽籤；每次執行的笑話可能都不同。
> - temperature 小 → 保守穩定；大 → 有創意但也更容易離題。
> - `temperature=1` 不改變分佈，想看到明顯差別可以試 0.2 和 1.5。

## 9. top_p：只在「最有可能的那一群」裡抽籤

**top_p**（又稱 nucleus sampling）的做法：

1. 把候選 token 依機率**由高到低**排列；
2. 從最高的開始累加機率，直到累計達到 `p`；
3. **只保留**這群 token，其餘丟掉，然後在這群裡抽籤。

所以 `top_p` 控制的是「**可考慮的詞元子集有多大**」（也就是你註解寫的意思）：

| top_p | 效果 |
|---|---|
| 小（如 0.5） | 只剩最有把握的少數幾個字，**更保守** |
| 大（如 0.95） | 保留較多候選，**更多樣** |
| 1 | 這一關**不裁切**，所有 token 都留著 |

> ⚠️ **還有一個隱藏的細節：** 打開取樣後，transformers 預設還會套用 `top_k = 50`（只考慮機率最高的 50 個 token）。這是我查 5.20.1 版原始碼看到的預設值；模型自己的生成設定可能覆蓋它。所以 `top_p=1` 並不等於「完全沒有任何篩選」。

In [ ]:
#===================================================================
#top_p(採樣)控制LLM可考慮的詞元子集大小
output = pipe(message, do_sample=True, top_p=1)
print(output[0]["generated_text"])
#===================================================================

> 📌 **本節重點**
> - top_p 控制「候選名單有多大」；temperature 控制「名單內的機率有多平」。
> - 兩者通常**擇一調整**就好，不必同時大幅修改。
> - 想要穩定、可重現 → `do_sample=False`；想要多樣 → `do_sample=True` 再調 temperature 或 top_p。

## 10. 情境學習（In-Context Learning）：在提示裡給範例

模型沒聽過 "Gigamuru"、"screeg" 這些**自創的詞**。我們用對話歷史**示範一次**：

| 順序 | role | 內容 |
|---|---|---|
| 1 | user | 說明 Gigamuru 是什麼，並請它造句 |
| 2 | assistant | **示範答案**：造好的句子 |
| 3 | user | 說明 screeg 是什麼，請它造句 ← **這次輪到模型回答** |

這就是 **one-shot**（給 1 個範例）。給 0 個叫 zero-shot，給多個叫 few-shot。模型從你給的範例**學到格式與做法**，不需要重新訓練。

### 小細節：字串怎麼「接」在一起？

第 1 則的 `content` 是兩個字串字面值寫在相鄰兩行，中間沒有逗號：

```python
"content" : "A 'Gigamuru' is ... instrument. "
"An example of a sentence ... is: "
```

Python 會自動把相鄰的字串字面值**接成一個字串**（所以每段結尾的空格要留著，不然兩段會黏在一起）。

In [ ]:
#===================================================================
#情境學習： 提供範例
one_shot_prompt = [
    {
        "role" : "user",
        "content" : "A 'Gigamuru' is a type of Japanese musical instrument. "
        "An example of a sentence that uses the word Gigamuru is: "
    },
    {
        "role" : "assistant",
        "content" : "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home. "
    },
    {
        "role" : "user",
        "content" : "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    },
]

print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))
output = pipe(one_shot_prompt)
print(output[0]["generated_text"])
#===================================================================

> 📌 **本節重點**
> - 在對話歷史裡放 `user → assistant` 的範例，模型會模仿你示範的格式。
> - 命名上它叫 `one_shot_prompt`，但它其實是一個**訊息 list**，不是字串。
> - 這裡先 `print` 模板、再生成：前者是「看」，後者才是真的生成。

## 11. 連鎖提示（Prompt Chaining）：用上一步的輸出當下一步的輸入

複雜任務拆成幾個小步驟，**前一個提示的輸出，放進後一個提示**：

```
 提示 1：幫聊天機器人取名字與口號
    │ 輸出存進 product_description
    ▼
 提示 2：根據「這個產品」寫一段推銷詞
```


In [ ]:
#===================================================================
#連鎖提示： 將一個prompt的輸出作為下一個promt的輸入
product_prompt = [
    {
        "role" : "user",
        "content" : "Create a name and slogan for a chatbot that leverages LLM."
    },
]

output = pipe(product_prompt)
product_description = output[0]['generated_text']
print(product_description)

sales_prompt = [
    {
        "role" : "user",
        "content" : f"Generate a very short sales pitch for the following product: {product_description}"
    },
]
output = pipe(sales_prompt)
sales_pitch = output[0]['generated_text']
print(sales_pitch)
#===================================================================

> 📌 **本節重點**
> - 連鎖提示 = 前一次的輸出，當作後一次輸入的一部分。
> - f-string 裡 `{變數}` 才會代入內容；`{'變數'}` 只會印出名字，而且**不會報錯**，所以很難發現。
> - 好習慣：生成之後，把「實際送出去的提示」印出來檢查一次。

## 12. 全課重點整理

### 技巧對照表

| 技巧 | 程式裡怎麼做 | 作用 |
|---|---|---|
| 基本生成 | `pipe(message)` | 傳訊息 list，取 `output[0]["generated_text"]` |
| 看聊天模板 | `apply_chat_template(..., tokenize=False)` | 看模型實際收到的文字 |
| temperature | `do_sample=True, temperature=...` | 調整機率分佈的平坦程度 |
| top_p | `do_sample=True, top_p=...` | 限制可選 token 的範圍 |
| 情境學習 | 在訊息裡放 `user → assistant` 範例 | 讓模型模仿格式與做法 |
| 連鎖提示 | 前一個輸出放進後一個提示 | 把複雜任務拆成小步驟 |

### 必記的 7 件事

1. LLM 一次預測一個 token；pipeline 把「套模板、分詞、生成、解碼」包成一步。
2. 訊息格式是 **list of dict**（`role` + `content`）。
3. 傳訊息 list 給 pipeline → 它自動套聊天模板；`apply_chat_template` 在這裡主要用來**看**。
4. `do_sample=False` 穩定可重現；`True` 才會抽籤，**temperature、top_p 才有作用**。
5. temperature 調「機率平不平」，top_p 調「候選名單多大」。
6. 情境學習靠對話歷史裡的範例，不需要訓練。
7. 連鎖提示要確認「上一步輸出」真的被放進下一個提示（f-string 別加引號）。

### 常見錯誤

| 現象 | 可能原因 |
|---|---|
| 參數沒效果，還出現警告 | `do_sample=False` 時 `temperature`、`top_p` 會被忽略 |
| CUDA out of memory | 顯存不足（約需 7.6 GB 以上） |
| 連鎖提示第二步答非所問 | f-string 寫成 `{'變數'}`，沒代入內容 |
| 印出來的是一串數字 | `apply_chat_template` 沒加 `tokenize=False` |
| 兩段字串黏在一起 | 相鄰字串字面值相接時，結尾忘了留空格 |
| 每次笑話都一樣 | 這是 `do_sample=False` 的正常行為 |

### 動手練習

1. 把 `temperature` 分別改成 0.2 和 1.5，各跑 3 次，比較笑話的差異。
2. 把 `top_p` 改成 0.5，觀察輸出變化。
3. 在 `apply_chat_template` 加上 `add_generation_prompt=True`，看結尾多了什麼。
4. 把情境學習改成 **zero-shot**（只留第 3 則 user 訊息），比較結果。
5. 加一則 `{"role": "system", "content": "..."}` 在訊息最前面，觀察模型語氣的變化。
6. 自己設計一個三步驟的連鎖提示（例如：取名 → 寫口號 → 寫推銷詞），並確認每一步都有被代入。

### 延伸方向

- 手動 `model.generate()`：自己做 `apply_chat_template → 分詞 → generate → decode`
- 其他取樣參數：`top_k`、`repetition_penalty`、`min_p`
- few-shot、chain-of-thought（讓模型一步一步想）